In [1]:
from torch_harmonics.spherical_harmonics import SphericalHarmonics

In [2]:
import numpy as np
import matplotlib.pyplot as plt
from torch_harmonics import plotting
from torch.utils.data import DataLoader
import xarray
import torch
from torch import nn

In [3]:
f = xarray.load_dataset('/home/colin/hdd/workspace/datasets/test_dataset.nc')

In [4]:
dataset = f['rti']
dataset = torch.tensor(dataset.to_numpy()[:,:,:,:,0]).view(-1,61,21,107).repeat(1,2,1,1)
dataset = dataset.permute(0,3,1,2)
dataset = dataset[:, 40:-40]

In [5]:
range_reduce = dataset[:, np.arange(0,27,2), :, :]
reduce = range_reduce[:,:,np.arange(0,122,4),:]

In [10]:
B = 8
dataloader = DataLoader(reduce, batch_size=B, shuffle=True)

In [7]:
next(iter(dataloader)).shape

torch.Size([32, 14, 31, 21])

In [8]:
class SHT(nn.Module):
    def __init__(self, L=3, num_theta=31, num_phi=21, num_range=14):
        super().__init__()
        self.num_range = num_range
        self.L = L
        self.w_lin = nn.Linear(num_range*num_theta*num_phi, num_range*L*(L+1)*2)
        self.sh = SphericalHarmonics(L, num_lat=num_theta, num_lon=num_phi)
        #self.mlp_out = nn.Linear(num_range*num_theta*num_phi, num_range*num_theta*num_phi)

    def forward(self, x, coords):
        B = x.size(0)
        
        w = self.w_lin(x)
        w = w.view(-1, self.L, self.L+1, 2)
        
        y = self.sh(w, coords).float()#.view(B, self.num_range)
        #y = self.mlp_out(y).view(B,self.sh.num_lat, self.sh.num_lon)
        return y.view(B, self.num_range, self.sh.num_lat, self.sh.num_lon), w.detach()

In [11]:
L = 75
sh_model = SHT(L).cuda()
optimizer = torch.optim.Adam(sh_model.parameters(), lr = 1e-4)
num_epochs = 100

for epoch in range(num_epochs):
    epoch_loss = 0.0
    for batch in dataloader:
        B = batch.size(0)
        batch = batch.cuda()

        a = torch.linspace(0, 2*np.pi, 21)
        b = torch.linspace(0, 2*np.pi, 32)
        a_coords = torch.randint(0, 20, (B,))
        b_coords = torch.randint(0, 31, (B,))
        coords = torch.vstack([
            b[b_coords],
            a[a_coords]
        ]).permute(1,0)
        coords = coords.repeat(1,1)
        #y_true = batch[torch.arange(B), [24]*B, b_coords, a_coords]
        #y_true = batch[:,24]

        y_true = batch
        y_pred, _ = sh_model(batch.view(B, -1), None)
        loss = (y_pred - y_true).pow(2).mean()
    
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()
    
    print(f"Loss: {epoch_loss / len(dataloader)}")

OutOfMemoryError: CUDA out of memory. Tried to allocate 5.42 GiB. GPU 

In [ ]:
with torch.no_grad():
    out, _ = sh_model.cpu()(batch[1].view(1,-1).cpu(), None)
plotting.plot_spherical_fn(out[0,10], fig=plt.figure(figsize=(2,2)))

In [ ]:
plotting.plot_spherical_fn(batch[1,10].cpu(), fig=plt.figure(figsize=(2,2)))